## LCEL 인터페이스


Runnable은 LangChain에서 프롬프트 템플릿, LLM 호출기, 출력 파서 등 다양한 컴포넌트를 연결하고 실행하는 방식을 표준화한 공통 인터페이스입니다.

사용자 정의 체인을 가능한 쉽게 만들 수 있도록, [`Runnable`](https://api.python.langchain.com/en/stable/runnables/langchain_core.runnables.base.Runnable.html#langchain_core.runnables.base.Runnable) 프로토콜을 구현했습니다. 

`Runnable` 프로토콜은 대부분의 컴포넌트에 구현되어 있습니다.

이는 표준 인터페이스로, 사용자 정의 체인을 정의하고 표준 방식으로 호출하는 것을 쉽게 만듭니다.
표준 인터페이스에는 다음이 포함됩니다.

- [`stream`](#stream): 응답의 청크를 스트리밍합니다.
- [`invoke`](#invoke): 입력에 대해 체인을 호출합니다.
- [`batch`](#batch): 입력 목록에 대해 체인을 호출합니다. (여러 작업을 그룹으로 묶어서 일괄 처리하는 방식입니다. 여러 개의 딕셔너리로 이루어진 리스트를 인자로 받아 각 딕셔너리에서 특정 키 값으로 사용합니다.)


비동기 메소드도 있습니다.

- [`astream`](#async-stream): 비동기적으로 응답의 청크를 스트리밍합니다.
- [`ainvoke`](#async-invoke): 비동기적으로 입력에 대해 체인을 호출합니다.
- [`abatch`](#async-batch): 비동기적으로 입력 목록에 대해 체인을 호출합니다.
- [`astream_log`](#async-stream-intermediate-steps): 최종 응답뿐만 아니라 발생하는 중간 단계를 스트리밍합니다.

In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01-Basic")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH01-Basic


LCEL 문법을 사용하여 chain 을 생성합니다.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ChatOpenAI 모델을 인스턴스화합니다.
model = ChatOpenAI()
# 주어진 토픽에 대한 농담을 요청하는 프롬프트 템플릿을 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대하여 3문장으로 설명해줘.")
# 프롬프트와 모델을 연결하여 대화 체인을 생성합니다.
chain = prompt | model | StrOutputParser()

c:\metacode_project\metacode_study\.metacode_study_may\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


## stream: 실시간 출력


이 함수는 `chain.stream` 메서드를 사용하여 주어진 토픽에 대한 데이터 스트림을 생성하고, 이 스트림을 반복하여 각 데이터의 내용(`content`)을 즉시 출력합니다. `end=""` 인자는 출력 후 줄바꿈을 하지 않도록 설정하며, `flush=True` 인자는 출력 버퍼를 즉시 비우도록 합니다. 

In [4]:
# chain.stream 메서드를 사용하여 '멀티모달' 토픽에 대한 스트림을 생성하고 반복합니다.
for token in chain.stream({"topic": "멀티모달"}):
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

멀티모달은 여러 가지 다른 형식의 정보를 결합하여 사용자에게 제공하는 시스템이다. 이는 텍스트, 이미지, 음성, 동영상 등 여러 형태의 데이터를 포함할 수 있으며, 다양한 감각을 활용해 정보를 전달한다. 멀티모달은 사용자들이 정보를 더 쉽게 이해하고 상호작용할 수 있도록 돕는다.

## invoke: 호출


`chain` 객체의 `invoke` 메서드는 주제를 인자로 받아 해당 주제에 대한 처리를 수행합니다.

In [5]:
# chain 객체의 invoke 메서드를 호출하고, 'ChatGPT'라는 주제로 딕셔너리를 전달합니다.
chain.invoke({"topic": "ChatGPT"})

'ChatGPT는 OpenAI가 개발한 대화형 인공지능 모델로 자연어 대화를 이해하고 응답합니다. 이 모델은 다양한 주제에 대한 정보를 학습하고 사용자와의 상호작용을 통해 학습을 이어가며 발전합니다. ChatGPT를 통해 사람과 유사한 자연스러운 대화를 경험할 수 있습니다.'

## batch: 배치(단위 실행)


함수 `chain.batch`는 여러 개의 딕셔너리를 포함하는 리스트를 인자로 받아, 각 딕셔너리에 있는 `topic` 키의 값을 사용하여 일괄 처리를 수행합니다.

In [6]:
# 주어진 토픽 리스트를 batch 처리하는 함수 호출
chain.batch([{"topic": "ChatGPT"}, {"topic": "Instagram"}])

['ChatGPT는 OpenAI의 고급 언어 모델로, 자연어 처리 및 이해 능력을 바탕으로 다양한 주제에 대해 대화할 수 있습니다. 인공지능이 질문에 답하고 이야기를 나누는 데 사용되며, 사용자의 선호도와 관심사에 맞게 맞춤형 응답을 제공합니다. 사용자와 자연스럽고 유익한 재미있는 대화를 경험할 수 있는 혁신적인 기술입니다.',
 '인스타그램은 사진, 동영상 및 스토리를 공유하는 소셜 미디어 플랫폼이다. 2010년에 출시된 이후 전 세계적으로 많은 이용자들을 보유하고 있으며, 시간이 지남에 따라 다양한 기능과 서비스를 추가했다. 인스타그램은 사용자들이 콘텐츠를 공유하고 소통할 수 있는 공간으로, 인기 있는 브랜드나 인플루언서들 또한 활발하게 활동하고 있다.']

`max_concurrency` 매개변수를 사용하여 동시 요청 수를 설정할 수 있습니다

`config` 딕셔너리는 `max_concurrency` 키를 통해 동시에 처리할 수 있는 최대 작업 수를 설정합니다. 여기서는 최대 3개의 작업을 동시에 처리하도록 설정되어 있습니다.

In [7]:
chain.batch(
    [
        {"topic": "ChatGPT"},
        {"topic": "Instagram"},
        {"topic": "멀티모달"},
        {"topic": "프로그래밍"},
        {"topic": "머신러닝"},
    ],
    config={"max_concurrency": 3},
)

['ChatGPT는 인공지능 챗봇으로 자연어 처리 기술을 이용하여 대화를 나눌 수 있는 서비스이다. 사용자의 질문이나 요청에 신속하고 정확하게 응답하여 다양한 주제에 대해 대화를 이끌어낼 수 있다. 인공지능 기술 발전의 한 예로, 사용자와 대화를 통해 정보를 공유하고 도움을 줄 수 있는 새로운 소통 수단이다.',
 'Instagram은 사용자들이 사진과 동영상을 공유하고 다른 사람들과 소통하는 소셜 미디어 플랫폼이다. 또한 해시태그를 통해 관심사나 주제별로 컨텐츠를 검색할 수 있으며, 팔로워를 통해 자신의 소식을 전달하고 다양한 커뮤니케이션을 할 수 있다. 또한 비즈니스 활동을 위한 프로모션과 광고 플랫폼으로도 활용되고 있다.',
 '멀티모달은 여러 가지 다른 형태의 정보를 결합하여 사용자에게 제공하는 시스템이다. 예를 들어, 음성, 텍스트, 이미지, 비디오 등 다양한 형태의 정보를 함께 사용할 수 있다. 멀티모달은 사용자 경험을 향상시키고 정보 전달을 보다 효과적으로 할 수 있도록 도와준다.',
 '프로그래밍은 컴퓨터에게 수행할 작업을 지시하는 일종의 언어입니다. 코드를 작성하여 프로그램을 만들고, 실행하여 원하는 결과를 얻을 수 있습니다. 각 언어마다 문법과 규칙이 있으며, 이를 잘 활용하여 문제를 해결하는 것이 프로그래밍의 핵심입니다.',
 '머신러닝은 컴퓨터 시스템이 데이터를 분석하고 학습하여 패턴을 발견하고 예측하는 인공지능 기술이다. 이를 통해 우리는 복잡한 문제를 해결하고 의사결정을 돕는데 활용할 수 있다. 머신러닝은 이미지 및 음성 인식, 추천 시스템, 자율주행 자동차 등 다양한 분야에서 혁신을 이루고 있다.']

## async stream: 비동기 스트림


함수 `chain.astream`은 비동기 스트림을 생성하며, 주어진 토픽에 대한 메시지를 비동기적으로 처리합니다.

비동기 for 루프(`async for`)를 사용하여 스트림에서 메시지를 순차적으로 받아오고, `print` 함수를 통해 메시지의 내용(`s.content`)을 즉시 출력합니다. `end=""`는 출력 후 줄바꿈을 하지 않도록 설정하며, `flush=True`는 출력 버퍼를 강제로 비워 즉시 출력되도록 합니다.


In [8]:
# 비동기 스트림을 사용하여 'YouTube' 토픽의 메시지를 처리합니다.
async for token in chain.astream({"topic": "YouTube"}):
    # 메시지 내용을 출력합니다. 줄바꿈 없이 바로 출력하고 버퍼를 비웁니다.
    print(token, end="", flush=True)

YouTube는 구글이 소유하고 운영하는 온라인 동영상 플랫폼으로, 사용자들은 다양한 주제의 동영상을 시청하고 업로드할 수 있다. 유명한 유튜버들이 채널을 운영하고 다양한 콘텐츠를 제공하여 수많은 시청자들을 끌어들이고 있다. 또한 광고를 통해 수익을 창출할 수 있는데, 이를 통해 수많은 유튜버들이 수익을 창출하고 있다.

## async invoke: 비동기 호출


`chain` 객체의 `ainvoke` 메서드는 비동기적으로 주어진 인자를 사용하여 작업을 수행합니다. 여기서는 `topic`이라는 키와 `NVDA`(엔비디아의 티커) 라는 값을 가진 딕셔너리를 인자로 전달하고 있습니다. 이 메서드는 특정 토픽에 대한 처리를 비동기적으로 요청하는 데 사용될 수 있습니다.


In [9]:
# 비동기 체인 객체의 'ainvoke' 메서드를 호출하여 'NVDA' 토픽을 처리합니다.
my_process = chain.ainvoke({"topic": "NVDA"})

In [10]:
# 비동기로 처리되는 프로세스가 완료될 때까지 기다립니다.
await my_process

'NVDA는 엔비디아(NVIDIA)라는 회사의 주식 코드이다. 엔비디아는 그래픽 처리 장치를 개발하고 제조하는 세계적인 기업으로, 게임 산업 뿐만 아니라 인공지능, 자율 주행 차량 등 다양한 분야에서 기술을 지원하고 있다. NVDA 주식은 기술 기업들의 성장에 대한 투자 수익을 노리는 투자자들 사이에서 인기가 높다.'

## async batch: 비동기 배치


함수 `abatch`는 비동기적으로 일련의 작업을 일괄 처리합니다.

이 예시에서는 `chain` 객체의 `abatch` 메서드를 사용하여 `topic` 에 대한 작업을 비동기적으로 처리하고 있습니다.

`await` 키워드는 해당 비동기 작업이 완료될 때까지 기다리는 데 사용됩니다.


In [11]:
# 주어진 토픽에 대해 비동기적으로 일괄 처리를 수행합니다.
my_abatch_process = chain.abatch(
    [{"topic": "YouTube"}, {"topic": "Instagram"}, {"topic": "Facebook"}]
)

In [12]:
# 비동기로 처리되는 일괄 처리 프로세스가 완료될 때까지 기다립니다.
await my_abatch_process

['YouTube는 전 세계적으로 가장 큰 동영상 공유 플랫폼으로, 수많은 사용자들이 다양한 콘텐츠를 업로드하고 시청할 수 있는 서비스입니다. 뮤직 비디오, 블로그, 게임 플레이 영상 등 다양한 장르의 동영상을 제공하며, 사용자들은 구독을 통해 원하는 채널을 지속적으로 시청할 수 있습니다. 또한 광고 수익을 통해 동영상 제작자들이 수익을 창출할 수 있는 플랫폼으로 인기를 얻고 있습니다.',
 '인스타그램은 사진 및 영상을 공유하고 다양한 필터와 효과를 적용해 다른 사용자와 소통할 수 있는 소셜 미디어 플랫폼이다. 사용자는 팔로우, 좋아요, 댓글 등의 기능을 통해 소통하며 자신의 취향에 맞는 콘텐츠를 탐색할 수 있다. 인기 있는 인플루언서와 브랜드는 마케팅에 활용하기도 하는데, 일상 생활 속에서 특별한 순간을 기록하고 공유하는 데에 널리 사용된다.',
 'Facebook은 전 세계적으로 가장 인기 있는 소셜 네트워킹 서비스 중 하나이며, 사용자들은 친구들과 가족들과 소통하고 사진 및 동영상을 공유할 수 있다. 또한 비즈니스 및 마케팅 활동에도 널리 활용되고 있다.']

## Parallel: 병렬성

LangChain 공식 문서에 따르면 Runnable이란 사용자가 원하는 형태의 체인을 쉽게 만들 수 있도록 지원하는 실행 가능한 프로토콜이라고 설명하고 있습니다. prompt, model, StrOutputParser()는 모두 Runnable 프로토콜을 가지고 있습니다.
이 요소들을 연결해 만든 chain 자체도 Runnable입니다. 이렇게 여러 Runnable 체인을 병렬적으로 동시에 실행하는 것을 RunnableParallel이라고 합니다. 

LangChain Expression Language가 병렬 요청을 지원하는 방법을 살펴봅시다.
예를 들어, `RunnableParallel`을 사용할 때, 각 요소를 병렬로 실행합니다.


`langchain_core.runnables` 모듈의 `RunnableParallel` 클래스를 사용하여 두 가지 작업을 병렬로 실행하는 예시를 보여줍니다.

`ChatPromptTemplate.from_template` 메서드를 사용하여 주어진 `country`에 대한 **수도** 와 **면적** 을 구하는 두 개의 체인(`chain1`, `chain2`)을 만듭니다.

이 체인들은 각각 `model`과 파이프(`|`) 연산자를 통해 연결됩니다. 마지막으로, `RunnableParallel` 클래스를 사용하여 이 두 체인을 `capital`와 `area`이라는 키로 결합하여 동시에 실행할 수 있는 `combined` 객체를 생성합니다.


In [13]:
from langchain_core.runnables import RunnableParallel

# {country} 의 수도를 물어보는 체인을 생성합니다.
chain1 = (
    PromptTemplate.from_template("{country} 의 수도는 어디야?")
    | model
    | StrOutputParser()
)

# {country} 의 면적을 물어보는 체인을 생성합니다.
chain2 = (
    PromptTemplate.from_template("{country} 의 면적은 얼마야?")
    | model
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성합니다.
combined = RunnableParallel(capital=chain1, area=chain2)

`chain1.invoke()` 함수는 `chain1` 객체의 `invoke` 메서드를 호출합니다.

이때, `country`이라는 키에 `대한민국`라는 값을 가진 딕셔너리를 인자로 전달합니다.


In [14]:
# chain1 를 실행합니다.
chain1.invoke({"country": "대한민국"})

'대한민국의 수도는 서울이다.'

이번에는 `chain2.invoke()` 를 호출합니다. `country` 키에 다른 국가인 `미국` 을 전달합니다.


In [15]:
# chain2 를 실행합니다.
chain2.invoke({"country": "미국"})

'미국의 면적은 약 9,830만 제곱 킬로미터입니다. (3,800만 제곱 마일)'

`combined` 객체의 `invoke` 메서드는 주어진 `country`에 대한 처리를 수행합니다.

이 예제에서는 `대한민국`라는 주제를 `invoke` 메서드에 전달하여 실행합니다.


In [16]:
# 병렬 실행 체인을 실행합니다.
combined.invoke({"country": "대한민국"})

{'capital': '대한민국의 수도는 서울이야.', 'area': '대한민국의 총 면적은 약 100,363km² 입니다.'}

### 배치에서의 병렬 처리

병렬 처리는 다른 실행 가능한 코드와 결합될 수 있습니다.
배치와 병렬 처리를 사용해 보도록 합시다.


`chain1.batch` 함수는 여러 개의 딕셔너리를 포함하는 리스트를 인자로 받아, 각 딕셔너리에 있는 "topic" 키에 해당하는 값을 처리합니다. 이 예시에서는 "대한민국"와 "미국"라는 두 개의 토픽을 배치 처리하고 있습니다.


In [17]:
# 배치 처리를 수행합니다.
chain1.batch([{"country": "대한민국"}, {"country": "미국"}])

['대한민국의 수도는 서울입니다.', '미국의 수도는 워싱턴 D.C.입니다.']

`chain2.batch` 함수는 여러 개의 딕셔너리를 리스트 형태로 받아, 일괄 처리(batch)를 수행합니다.

이 예시에서는 `대한민국`와 `미국`라는 두 가지 국가에 대한 처리를 요청합니다.


In [18]:
# 배치 처리를 수행합니다.
chain2.batch([{"country": "대한민국"}, {"country": "미국"}])

['대한민국의 총 면적은 약 100,363 제곱 킬로미터입니다.', '미국의 면적은 약 9,833,520 평밀입니다.']

`combined.batch` 함수는 주어진 데이터를 배치로 처리하는 데 사용됩니다. 이 예시에서는 두 개의 딕셔너리 객체를 포함하는 리스트를 인자로 받아 각각 `대한민국`와 `미국` 두 나라에 대한 데이터를 배치 처리합니다.


In [19]:
# 주어진 데이터를 배치로 처리합니다.
combined.batch([{"country": "대한민국"}, {"country": "미국"}])

[{'capital': '대한민국의 수도는 서울입니다.', 'area': '대한민국의 면적은 약 100,363 제곱 킬로미터 입니다.'},
 {'capital': '미국의 수도는 워싱턴 D.C.입니다.', 'area': '미국의 면적은 약 9,826,675 km² 입니다.'}]